# 실제 따릉이 사례 → 그래프 → QoolQit QPU MIS

로컬 자료 처리 결과와 실제 Pasqal QPU 제출 코드를 연결한 노트북 파일

1. data_process.py가 원자료로 사례와 고전 난도 측정을 생성합니다.
2. graph.json은 후보 충돌, exact_graph.json은 생성 가능한 경우 전체 선언 제약의 정확 인코딩입니다.
3. 원자 배치는 별도 문제입니다. 수치 배치가 원래 MIS를 보존하는지 검사합니다.
4. QPU 제출과 결과 조회는 기본적으로 꺼져 있습니다. 원격 제출은 비용이 발생합니다.

SDK 환경: **.venv-qpu**. 필요 시 이 환경에 ipykernel을 설치하여 Jupyter 커널로 등록하세요.
설치 명령: **.venv-qpu/Scripts/python -m pip install -r requirements-qpu.txt**.


In [ ]:
# 자격 증명은 프로젝트 루트의 .env 파일에서 읽습니다. 기본값에서는 인증도 실행하지 않습니다.
ALLOW_PASQAL_CONNECTION = False

# 연결 객체의 기본값은 None입니다.
connection = None

# 사용자가 명시적으로 켠 경우에만 .env를 읽어 Pasqal Cloud에 연결합니다.
if ALLOW_PASQAL_CONNECTION:
    import os
    from pathlib import Path
    from dotenv import load_dotenv
    from pasqal_cloud import PasqalCloudConnection

    # .env의 값을 환경 변수로 불러옵니다.
    load_dotenv(Path.cwd() / ".env")

    connection = PasqalCloudConnection(
        username=os.environ["PASQAL_USERNAME"],      # Pasqal Cloud 플랫폼의 사용자 계정(이메일)
        password=os.environ["PASQAL_PASSWORD"],      # Pasqal Cloud 플랫폼 비밀번호
        project_id=os.environ["PASQAL_PROJECT_ID"],  # 계정에 연결된 프로젝트 ID
    )

## 1. 로컬 실행 결과

저장된 파일만 읽습니다. 실제 요청·후보 수, 그래프 규모, 제한시간과 최적성 격차를 함께 확인합니다.
‘stress’나 후보 수 증가만으로 고전적으로 어렵다고 단정하지 않습니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# 작업 폴더가 프로젝트 루트인지 확인합니다.
ROOT = Path.cwd()
if not (ROOT / "origin_data").is_dir():
    raise FileNotFoundError("Jupyter 작업 폴더를 quantum_bike로 설정하세요.")

# 로컬 실행 결과 요약 파일이 있는지 확인합니다.
SUMMARY_PATH = ROOT / "outputs" / "summary.csv"
if not SUMMARY_PATH.exists():
    raise FileNotFoundError("먼저 로컬 data_process.py를 실행해 사례를 생성하세요.")

# 요약 표를 불러와 표시합니다.
summary = pd.read_csv(SUMMARY_PATH)
display(summary)


## 2. 사례와 그래프 선택

별도 qpu_demo 출력이 있으면 먼저 선택합니다. **실제 요청에서 뽑은 작은 작동 확인 사례이며 고난도 사례가 아닙니다.**
CASE_OUTPUT을 outputs로 바꾸면 본 실험 사례를 선택할 수 있습니다.

직접 후보 그래프가 선언된 운영 제약과 정확히 같으면 graph.json을 사용하고, 그렇지 않으면 exact_graph.json을 우선 확인합니다.
정확 인코딩에는 보조 노드가 늘어날 수 있으며 장치 배치 가능성을 별도로 검사합니다.
현재 summary.csv에 기록된 사례만 선택하며, 그래프의 case_sha256와 현재 case.json 바이트 해시가 일치해야 합니다. 이전 실행의 남은 파일은 사용하지 않습니다.

In [ ]:
from bike_research.qpu import load_case_bound_graph

# qpu_demo 출력이 있으면 먼저 쓰고, 없으면 본 실험 출력을 사용합니다.
CASE_OUTPUT = ROOT / "outputs" / "qpu_demo"
if not (CASE_OUTPUT / "summary.csv").exists():
    CASE_OUTPUT = ROOT / "outputs"
print("선택한 사례 묶음:", CASE_OUTPUT.name)

# 현재 summary.csv에 기록된 사례 ID 목록을 읽습니다.
current_summary = pd.read_csv(CASE_OUTPUT / "summary.csv", dtype={"case_id": str})
case_ids = current_summary["case_id"].dropna().drop_duplicates().tolist()
if not case_ids:
    raise FileNotFoundError("현재 실행의 summary.csv에 사례가 없습니다.")

# 사용할 사례를 고릅니다.
CASE_ID = case_ids[0]  # 반드시 현재 summary.csv의 사례 ID 중에서 선택
if CASE_ID not in case_ids:
    raise ValueError("현재 실행 결과에 포함되지 않은 사례입니다.")

# 후보 충돌 그래프를 사례 해시와 대조해 읽습니다.
CASE_DIR = CASE_OUTPUT / "cases" / CASE_ID
GRAPH_PATH = CASE_DIR / "graph.json"
graph_input = load_case_bound_graph(GRAPH_PATH)

# 선언된 제약과 동치가 아니면, 정확 인코딩에 성공한 경우 그 그래프로 바꿉니다.
if graph_input.source.get("metadata", {}).get("representation") != "exact_declared_constraints":
    status_path = CASE_DIR / "encoding_status.json"
    status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
    if status.get("status") == "encoded":
        GRAPH_PATH = CASE_DIR / "exact_graph.json"
        graph_input = load_case_bound_graph(GRAPH_PATH)

# 그래프 문서와 사례 문서를 준비합니다.
graph_document = graph_input.source
case_document = json.loads((CASE_DIR / "case.json").read_text(encoding="utf-8"))

# 선택 결과를 표시합니다.
display(pd.DataFrame([graph_document.get("metadata", {})]))
print("선택 파일:", GRAPH_PATH.name)
print("노드:", len(graph_document["nodes"]), "엣지:", len(graph_document["edges"]))


## 3. 로컬 원자 배치와 비가중 MIS 펄스 준비

RUN_LOCAL_PREPARATION=True는 로컬 계산만 실행합니다. 인증이나 QPU 작업은 발생하지 않습니다.
외부 원자 좌표가 있으면 COORDINATES_PATH에 지정합니다. JSON 형식은
{"coordinates": {"노드ID": [x, y], ...}}이며 QoolQit의 무차원 좌표입니다.

원래 노드와 엣지의 보존, 비엣지 상호작용까지 포함한 최종 Hamiltonian의 충분조건을 확인합니다.
실패 시 엣지를 삭제하거나 에뮬레이터로 자동 대체하지 않습니다. 일반 그래프의 정확한 장치용 환원은 별도 과제입니다.


In [ ]:
from bike_research.qpu import prepare_qpu_program

# 로컬 준비 실행 여부와 입력 설정입니다.
RUN_LOCAL_PREPARATION = False
ALLOW_PAIRWISE_RELAXATION = False  # 그래프 자체의 MIS 실험임을 명시적으로 인정할 때만 변경
COORDINATES_PATH = None
prepared = None

# 켜져 있을 때만 로컬에서 배치와 펄스를 준비합니다. 인증과 QPU 호출은 없습니다.
if RUN_LOCAL_PREPARATION:
    prepared = prepare_qpu_program(
        GRAPH_PATH,
        coordinates_path=COORDINATES_PATH,
        allow_pairwise_relaxation=ALLOW_PAIRWISE_RELAXATION,
        seed=42,
        maxiter=5000,
        max_embedding_nodes=80,  # 로컬 탐색 비용 한도. QPU 보장 용량이 아님
        duration=40.0,          # 무차원 파일럿 파라미터. 최적 schedule 주장 없음
    )

    # 배치 검증 결과와 원자 배치를 표시합니다.
    display(pd.DataFrame([prepared.embedding_report]))
    prepared.program.register.draw()
else:
    print("로컬 QoolQit 준비: 실행하지 않음")


## 4. 실제 QPU 제출 — 비용 발생 구간

기존 인증 셀의 ALLOW_PASQAL_CONNECTION과 아래 제출 설정을 사용자가 명시적으로 켜기 전에는 원격 호출이 없습니다.
장치 이름은 계정의 실제 접근 권한과 맞아야 합니다. 컴파일 실패 시 제출하지 않으며 자동 재시도하지 않습니다.
제출 기록 파일이 이미 있으면 중복 제출을 막습니다. 실패 기록의 상태가 불명확하면 포털에서 확인하세요.

이 코드는 **qoolqit.execution.QPU**를 사용합니다. 로컬/원격 에뮬레이터를 사용하지 않습니다.

실행시간은 장치 최대 허용 시간의 80%로 명시적으로 조정하는 파일럿 설정입니다. 이 설정과 실제 컴파일 시간을 제출 기록에 남기며, 최적 schedule이라는 뜻은 아닙니다. 컴파일된 C6·좌표·최종 detuning을 다시 검사한 뒤에만 제출합니다.

In [ ]:
from bike_research.qpu import submit_qpu

# 제출 설정입니다. 제출 스위치의 기본값은 꺼짐입니다.
ENABLE_PAID_QPU_SUBMISSION = False
DEVICE_NAME = "FRESNEL"  # 실제 계정의 장치 접근 여부는 제출 준비 시 확인됨
NUM_SHOTS = 100
DEVICE_DURATION_RATIO = 0.8  # 장치 허용 시간의 80%로 명시적 조정; 품질 보장 없음
SUBMISSION_RECEIPT = CASE_DIR / "qpu_submission_001.json"
job = None

# 켜져 있을 때만 실제 QPU에 제출합니다. 비용이 발생합니다.
if ENABLE_PAID_QPU_SUBMISSION:
    # 로컬 준비와 인증이 끝났는지 확인합니다.
    if prepared is None:
        raise RuntimeError("먼저 로컬 배치 검증을 완료하세요.")
    if connection is None:
        raise RuntimeError("사용자가 명시적으로 Pasqal 인증 셀을 활성화해야 합니다.")

    # 제출하고 제출 기록 파일의 이름을 표시합니다.
    job = submit_qpu(
        prepared, connection,
        device_name=DEVICE_NAME,
        num_shots=NUM_SHOTS,
        authorize_paid_submission=ENABLE_PAID_QPU_SUBMISSION,
        receipt_path=SUBMISSION_RECEIPT,
        device_max_duration_ratio=DEVICE_DURATION_RATIO,
    )
    print("QPU 제출 기록:", SUBMISSION_RECEIPT.name)
else:
    print("QPU 제출: 꺼짐 — 과금 작업을 실행하지 않음")


## 5. 제출된 결과 조회와 원모형 검증

결과 조회는 제출과 별도입니다. job이 이미 생성된 경우에만 실행하며 새 작업을 자동 제출하지 않습니다.
원시 표본을 수선하지 않고 독립성·선택 수·운영 제약을 검사합니다.
정확 인코딩의 보조 노드 수를 추천 개수로 사용하지 않습니다.

커널을 재시작했다면 RECOVER_EXISTING_QPU_JOB을 사용자가 활성화하여 제출 기록의 job_id·batch_id로 기존 작업을 조회할 수 있습니다. 그래프·사례 해시와 비트 순서가 다르면 조회 전에 중단합니다.

In [ ]:
from bike_research.qpu import (
    evaluate_bitstrings, load_case_bound_graph, load_submission_receipt, recover_qpu_job,
)
from bike_research.graph import validate_selection
from bike_research.encoding import decode_operational_mis


def decode_candidates(selected_graph_ids):
    """그래프 노드 ID를 후보 ID로 복호화합니다. 직접 후보 그래프면 그대로 반환합니다."""
    if graph_document.get("encoding_map"):
        decoded = decode_operational_mis(graph_document, selected_graph_ids)
        return decoded["selected_candidate_ids"]
    return selected_graph_ids


# 커널 재시작 시 이미 제출된 job을 복구하는 조회. 새로 제출하지 않습니다.
RECOVER_EXISTING_QPU_JOB = False
if RECOVER_EXISTING_QPU_JOB:
    job = recover_qpu_job(
        GRAPH_PATH, SUBMISSION_RECEIPT, connection,
        authorize_remote_lookup=RECOVER_EXISTING_QPU_JOB,
    )

# 켜져 있을 때만 기존 job의 결과를 가져와 검증합니다.
FETCH_EXISTING_QPU_RESULTS = False
if FETCH_EXISTING_QPU_RESULTS:
    if job is None:
        raise RuntimeError("기존 job이 필요합니다. 재시작했다면 위 복구 조회를 활성화하세요.")

    # 그래프와 제출 기록을 다시 읽어 메모리의 job과 대조합니다.
    result_graph = load_case_bound_graph(GRAPH_PATH)
    receipt = load_submission_receipt(SUBMISSION_RECEIPT, result_graph)
    if job.job_id() != receipt["job_id"]:
        raise RuntimeError("메모리의 job과 선택한 제출 기록이 다릅니다. 기존 작업을 다시 복구하세요.")
    graph_document = result_graph.source
    case_document = json.loads((CASE_DIR / "case.json").read_text(encoding="utf-8"))

    # 측정 결과를 가져와 수선 없이 평가합니다.
    results = job.results(timeout=30)
    rows = evaluate_bitstrings(
        result_graph,
        results.final_bitstrings,
        node_order=receipt["node_order"],
        candidate_decoder=decode_candidates,
        operational_validator=lambda ids: validate_selection(case_document, ids),
    )

    # 정확 인코딩 그래프면 표본별 복호화 검증을 덧붙입니다.
    if graph_document.get("encoding_map"):
        for row in rows:
            row["encoding_validation"] = decode_operational_mis(
                graph_document, row["selected_nodes"]
            )

    # 평가 결과를 저장하고 상위 20개를 표시합니다.
    result_path = CASE_DIR / "qpu_samples_001.json"
    result_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    display(pd.DataFrame(rows).head(20))
else:
    print("QPU 결과 조회: 실행하지 않음")


## 근거와 해석 범위

- [QoolQit 실행 공식 문서](https://docs.pasqal.com/qoolqit/fundamentals/execution/execution/): 실제 QPU 클래스, 장치 컴파일, 결과 조회.
- [QoolQit embedding](https://docs.pasqal.com/qoolqit/fundamentals/embedding/): 수치 배치 방법.
- [MIS/MWIS 공식 예제](https://docs.pasqal.com/qoolqit/tutorials/Solving_a_MWIS/): 그래프·상호작용·펄스 연결. 여기서는 모든 노드의 가치를 1로 둡니다.

충분조건은 이상적 최종 Hamiltonian에 대한 논리 검사입니다. 노이즈·유한 실행시간·장치 오차 아래 최적해 획득 보장이 아닙니다.
실제 QPU 실행·접근 권한·측정 성능은 아직 검증하지 않았습니다.
